# Method 1

In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("cleaned_numeric_clinvar_dbnsfp_final.csv")

X = df.drop(columns=["CLNSIG"])
y = df["CLNSIG"].astype(int)

print(X.shape)  # (8508, 28)


(8508, 28)


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

importance = (
    pd.Series(rf.feature_importances_, index=X.columns)
      .sort_values(ascending=False)
)

importance.head(10)


EXAC_AF     0.214508
METARNN     0.143750
CLINPRED    0.141833
CADD        0.089920
GMVP        0.077129
METASVM     0.060365
VEST4       0.046262
REVEL       0.035547
MVP         0.025757
VARITY_R    0.020570
dtype: float64

In [7]:
top20 = importance.head(20).index.tolist()

X_top20 = X[top20]

print("Top-20 shape:", X_top20.shape)


Top-20 shape: (8508, 20)


In [8]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_top20_scaled = scaler.fit_transform(X_top20)

def pca_reduce(X, n):
    pca = PCA(n_components=n, random_state=42)
    return pca.fit_transform(X), pca.explained_variance_ratio_.sum()

pca_dims = [4, 8, 12, 16]

pca_results = {}
for d in pca_dims:
    X_pca, var = pca_reduce(X_top20_scaled, d)
    pca_results[d] = (X_pca, var)
    print(f"PCA-{d}: explained variance = {var:.3f}")


PCA-4: explained variance = 0.808
PCA-8: explained variance = 0.905
PCA-12: explained variance = 0.959
PCA-16: explained variance = 0.989


In [9]:
for d, (Xp, _) in pca_results.items():
    df_pca = pd.DataFrame(Xp, columns=[f"PC{i+1}" for i in range(d)])
    df_pca["CLNSIG"] = y.values
    df_pca.to_csv(f"clinvar_pca_top20_{d}.csv", index=False)


# Method 2

In [10]:
top_sets = {
    4: importance.head(4).index.tolist(),
    8: importance.head(8).index.tolist(),
    12: importance.head(12).index.tolist(),
    16: importance.head(16).index.tolist()
}

for k, feats in top_sets.items():
    print(f"Top-{k} features:", feats)


Top-4 features: ['EXAC_AF', 'METARNN', 'CLINPRED', 'CADD']
Top-8 features: ['EXAC_AF', 'METARNN', 'CLINPRED', 'CADD', 'GMVP', 'METASVM', 'VEST4', 'REVEL']
Top-12 features: ['EXAC_AF', 'METARNN', 'CLINPRED', 'CADD', 'GMVP', 'METASVM', 'VEST4', 'REVEL', 'MVP', 'VARITY_R', 'SIFT4G', 'PROVEAN']
Top-16 features: ['EXAC_AF', 'METARNN', 'CLINPRED', 'CADD', 'GMVP', 'METASVM', 'VEST4', 'REVEL', 'MVP', 'VARITY_R', 'SIFT4G', 'PROVEAN', 'M-CAP', 'PRIMATEAI', 'MUTATIONASSESSOR', 'DEOGEN2']


In [11]:
from sklearn.preprocessing import MinMaxScaler

for k, feats in top_sets.items():
    scaler = MinMaxScaler()
    Xk = scaler.fit_transform(X[feats])

    df_k = pd.DataFrame(Xk, columns=feats)
    df_k["CLNSIG"] = y.values

    df_k.to_csv(f"clinvar_top{k}_features.csv", index=False)
